<a href="https://colab.research.google.com/github/ekyuho/AI-on-the-edge-device/blob/rolling/fine_tune0523.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [2]:
import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import torch


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:

# 1. 프롬프트 템플릿 정의
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# ChatML 형식 템플릿 (대화형에 더 적합)
chatml_prompt = """<|im_start|>system
You are a helpful AI assistant.
<|im_end|>
{}"""


In [4]:

# 2. Multi-turn 대화를 단일 문자열로 변환하는 함수
def format_conversations_alpaca(examples):
    """Alpaca 형식으로 대화를 포맷팅"""
    texts = []

    for conversation in examples["conversations"]:
        formatted_text = ""

        for i in range(0, len(conversation), 2):
            if i + 1 < len(conversation):
                human_msg = conversation[i]["value"]
                assistant_msg = conversation[i + 1]["value"]

                # 첫 번째 턴
                if i == 0:
                    formatted_text = alpaca_prompt.format(
                        human_msg,
                        "",  # input은 비워둠
                        assistant_msg
                    )
                # 이후 턴들은 이어붙임
                else:
                    formatted_text += f"\n\n### Instruction:\n{human_msg}\n\n### Response:\n{assistant_msg}"

        texts.append(formatted_text + "<|endoftext|>")

    return {"text": texts}

def format_conversations_chatml(examples):
    """ChatML 형식으로 대화를 포맷팅"""
    texts = []

    for conversation in examples["conversations"]:
        formatted_text = "<|im_start|>system\nYou are a helpful AI assistant.\n<|im_end|>\n"

        for turn in conversation:
            role = "user" if turn["from"] == "human" else "assistant"
            formatted_text += f"<|im_start|>{role}\n{turn['value']}\n<|im_end|>\n"

        texts.append(formatted_text)

    return {"text": texts}


In [5]:

# 3. 데이터셋 로드 및 전처리
def prepare_dataset(data_path, format_type="alpaca"):
    """JSON 파일에서 데이터셋 로드 및 전처리"""

    # JSON 파일 로드
    with open(data_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Hugging Face Dataset으로 변환
    dataset = Dataset.from_list(data)

    # 대화 포맷팅
    if format_type == "alpaca":
        dataset = dataset.map(format_conversations_alpaca, batched=True)
    else:
        dataset = dataset.map(format_conversations_chatml, batched=True)

    return dataset


In [6]:

# 4. 모델 및 토크나이저 로드
def load_model_and_tokenizer(model_name="unsloth/llama-3-8b-bnb-4bit"):
    """Unsloth 모델 로드"""

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_name,
        max_seq_length=2048,  # Multi-turn은 길이가 길 수 있음
        dtype=None,
        load_in_4bit=True,
    )

    # LoRA 어댑터 추가
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )

    return model, tokenizer


In [7]:

# 5. 트레이닝 설정
def train_model(model, tokenizer, dataset, output_dir="/content/drive/MyDrive/kyuho/fine_tuned_model"):
    """모델 트레이닝"""

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=2048,
        dataset_num_proc=2,
        packing=False,  # Multi-turn은 packing 비추천
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            num_train_epochs=3,
            learning_rate=2e-4,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=10,
            optim="adamw_8bit",
            weight_decay=0.01,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir=output_dir,
            save_strategy="epoch",
            save_total_limit=2,
        ),
    )

    # 트레이닝 시작
    trainer.train()

    return trainer


In [8]:

# 6. 전체 파이프라인
def main():
    # 데이터셋 준비
    dataset = prepare_dataset("/content/drive/MyDrive/kyuho/finetune.dat", format_type="alpaca")
    print(f"데이터셋 크기: {len(dataset)}")
    print(f"첫 번째 예시:\n{dataset[0]['text'][:500]}...")

    # 모델 로드
    model, tokenizer = load_model_and_tokenizer()

    # 트레이닝
    trainer = train_model(model, tokenizer, dataset)

    # 모델 저장
    model.save_pretrained("final_model")
    tokenizer.save_pretrained("final_model")

    # LoRA 어댑터만 저장 (용량 절약)
    model.save_pretrained_merged("model_merged", tokenizer, save_method="lora")

    print("Fine-tuning 완료!")


In [9]:

# 7. 추론 테스트
def test_inference(model_path="final_model"):
    """Fine-tuned 모델로 추론 테스트"""

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    # 테스트 입력
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                "Python에서 딕셔너리를 정렬하는 방법을 알려주세요.",
                "",
                ""
            )
        ],
        return_tensors="pt"
    ).to("cuda")

    # 생성
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    response = tokenizer.batch_decode(outputs)[0]

    print("모델 응답:")
    print(response)


In [18]:

# 8. 데이터셋 검증 함수
def validate_dataset(data_path):
    """데이터셋 형식 검증"""
    with open(data_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    issues = []

    for idx, item in enumerate(data):
        if "conversations" not in item:
            issues.append(f"Item {idx}: 'conversations' 키 없음")
            continue

        convs = item["conversations"]
        if len(convs) % 2 != 0:
            issues.append(f"Item {idx}: 대화 턴 수가 홀수 ({len(convs)})")

        for i, turn in enumerate(convs):
            if "from" not in turn or "value" not in turn:
                issues.append(f"Item {idx}, Turn {i}: 필수 키 누락")
            elif turn["from"] not in ["human", "assistant"]:
                issues.append(f"Item {idx}, Turn {i}: 잘못된 from 값: {turn['from']}")
            elif False and i % 2 == 0 and turn["from"] != "human":
                issues.append(f"Item {idx}, Turn {i}: human 턴이어야 함")
                print(convs)
                break
            elif False and i % 2 == 1 and turn["from"] != "assistant":
                issues.append(f"Item {idx}, Turn {i}: assistant 턴이어야 함")
                print(convs)
                break

    if issues:
        print("데이터셋 검증 실패:")
        for issue in issues[:10]:  # 처음 10개만 출력
            print(f"  - {issue}")
        print(f"  ... 총 {len(issues)}개 문제 발견")
    else:
        print("데이터셋 검증 성공!")

    return len(issues) == 0


In [19]:

if __name__ == "__main__":
    # 데이터셋 검증
    if validate_dataset("/content/drive/MyDrive/Temp/finetune.dat"):
        # 메인 트레이닝 실행
        main()

        # 추론 테스트
        #test_inference()

데이터셋 검증 성공!


Map:   0%|          | 0/5581 [00:00<?, ? examples/s]

데이터셋 크기: 5581
첫 번째 예시:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
너는 오늘 기분이 어때?

### Input:


### Response:
저는 할머니랑 대화할 수 있어서 기분이 참 좋아요!

### Instruction:
나도 너랑 얘기하니까 기분이 한결 좋아진다.

### Response:
네.<|endoftext|>...
==((====))==  Unsloth 2025.5.7: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ewcapston2 (ewcapston2-ewha-womans-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.589500
20,1.443400
30,1.289100
40,1.234700
50,1.258700
60,1.240400
70,1.189000
80,1.201900
90,1.197200
100,1.165700


KeyboardInterrupt: 